In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import NamedTuple
import matplotlib.pyplot as plt
from scipy.stats import gamma, poisson

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

#### EXERCISE 1: VALIDATION

Pasted from exercise 4a)

In [ ]:
class Observation(NamedTuple):  # bad name
    delta_S: float
    # delta_I: float # is not included!
    delta_R: float

class SIRAmounts(NamedTuple):  
    S: float
    I: float
    R: float

def simulate_sir_simple_purepy(lam, mu, I_0, T: int = 100):
    """
    lam: ~ infection_rate
    mu: ~ 
    I_0: # infected at T=0
    T: simulations steps (f.e. days)
    

    return teh sequence of observations:
    x = [(delta_S(1), delta_R(1)), ... , (delta_S(T), delta_R(T))]
    and
    sequence of corresponding ODE variables.
    c = [(S(1), I(1), R(1)) , ... , (S(T ), I(T ), R(T )) ]

    

    hidden parameters: y = [lam,mu,I0]
    """

    # init variables
    I = I_0
    S = 900 # why are theese not function params???
    R = 10

    N = I + S + R

    observations:list[Observation] = [] # bad name
    amounts:list[SIRAmounts] = [] # bad name

    for T_i in range(T):
        delta_S = -lam * S * I / N
        delta_I = lam * S * I / N - mu * I
        delta_R = mu * I

        S += delta_S
        I += delta_I
        R += delta_R


        obs = Observation(delta_S ,delta_R)
        variables = SIRAmounts(S,I,R)
        observations.append(obs)
        amounts.append(variables)
    return observations, amounts
    

def simulate_sir_simple(lam, mu, I_0, T: int = 100):
    """
    numpy optimized version of the simulation that supports batches
    lam:ndarray | float~ infection_rate
    mu:ndarray | float ~ 
    I_0:ndarray | flaot   #infected at T=0
    T: simulations steps (f.e. days)
    

    expects the size of lam and mu and I_0 to be the same: (N,)

    return the sequence of observations:
    x = [(delta_S(1), delta_R(1)), ... , (delta_S(T), delta_R(T))]
    and
    sequence of corresponding ODE variables.
    c = [(S(1), I(1), R(1)) , ... , (S(T ), I(T ), R(T )) ]

    

    hidden parameters: y = [lam,mu,I0]
    """

    # init variables
    dtype = np.float32

    
    lam = np.array(lam,dtype=dtype)
    mu = np.array(mu,dtype=dtype)

    batch_size = 1 if lam.shape == () else lam.shape[0]

    I = np.array(I_0, dtype=dtype).reshape((batch_size,))
    S = np.array([900] * batch_size ,dtype=dtype) # why are theese not function params???
    R = np.array([10] * batch_size,dtype=dtype) # does not matter for now how many

    N = I + S + R

    observations = np.zeros((batch_size,T,2),dtype=dtype)
    amounts = np.zeros((batch_size,T,3),dtype=dtype) 

    for i in range(T):
        delta_S = -lam * S * I / N
        delta_I = lam * S * I / N - mu * I
        delta_R = mu * I

        S += delta_S
        I += delta_I
        R += delta_R


        #obs = Observation(delta_S ,delta_R)
        observations[:,i,:] = np.array([delta_S,delta_R]).T
        # variables = SIRAmounts(S,I,R)
        amounts[:,i,:] = np.array((S,I,R)).T
        
    return observations, amounts

In [ ]:
#alternative prior


lam_mean = 0.15
lam_std  = 0.05


mu_mean = 0.1
mu_std  = 0.025


I0_mean = 15

n_samples = 1000



def gamma_from_mean_std(mean, std):
    shape = (mean / std) ** 2    
    scale = (std ** 2) / mean     
    return shape, scale

 
lam_k, lam_theta = gamma_from_mean_std(lam_mean, lam_std)
mu_k,  mu_theta  = gamma_from_mean_std(mu_mean,  mu_std)



def sample_from_prior(size):
    lam_samples = gamma.rvs(a=lam_k, scale=lam_theta, size=size)
    mu_samples  = gamma.rvs(a=mu_k,  scale=mu_theta,  size=size)
    I0_samples  = poisson.rvs(mu=I0_mean, size=size)

    I0_samples = np.maximum(I0_samples, 1)

    return np.column_stack([lam_samples, mu_samples, I0_samples])


def visualize_prior(size):
    samples = sample_from_prior(size)

    plt.scatter(samples[:,0], samples[:,1], alpha=0.5)
    plt.xlabel("lambda")
    plt.ylabel("mu")
    plt.title("Gamma-Prior: lambda vs mu")
    plt.show()

    plt.hist(samples[:,2], bins=20)
    plt.xlabel("I0")
    plt.ylabel("count")
    plt.title("Poisson-Prior: I_0")
    plt.show()

In [ ]:
class RegressionNetwork(nn.Module):
    def __init__(self, input_size=200, hidden_size=400, layers=6, output_size = 3):
        """
        
        """
        super().__init__()
        self.input_size = input_size
        self.layers = layers
        self.encoder_layers = []
        self.encoder_layers.append((nn.Linear(input_size,hidden_size)))
        self.encoder_layers.append(nn.ReLU())
        for i in range(layers):
            self.encoder_layers.append(nn.Linear(hidden_size,hidden_size))
            self.encoder_layers.append(nn.ReLU())
        self.encoder_layers.append(nn.Linear(hidden_size,output_size))

        self.encoder = nn.Sequential(*self.encoder_layers)

    def forward(self,x):
        return self.encoder(x)

def train_regression_network(model,batch_size=64, T=100, epochs=500, learning_rate=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        #Sample new hidden parameters for this batch
        prior_samples = sample_from_prior(batch_size)
        lam = prior_samples[:,0]
        mu  = prior_samples[:,1]
        I0  = prior_samples[:,2]

        # Simulate SIR for the batch
        observations, _ = simulate_sir_simple(lam, mu, I0, T=T)

        X = observations.reshape(batch_size, -1)

        #hidden parameters
        y = np.column_stack([lam, mu, I0])

        X_torch = torch.tensor(X, dtype=torch.float32)
        y_torch = torch.tensor(y, dtype=torch.float32)

        # 6. Forward pass
        pred = model(X_torch)
        loss = criterion(pred, y_torch)

        # 9. Backprop + optimizer step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
# task 3:
# ex 3 copy of realnvp

# define RealNVP, and its building blocks
# device that is either cpu, cuda or mps
from typing import Optional

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")
#device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

def to_onehot(y,num_classes:int = 2):
    return torch.nn.functional.one_hot(y, num_classes=num_classes)


def get_orthonormal_matrix(d:int) -> torch.Tensor:
    A = torch.randn((d,d))
    Q, _R = torch.linalg.qr(A)
    return Q.to(device)

class CouplingBlock(nn.Module):
    def __init__(self, input_size:int, hidden_size:int, condition_size:int = 0):
        super(CouplingBlock, self).__init__()
        self.scale_net = nn.Sequential(
            nn.Linear(input_size - input_size//2 + condition_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size//2),
            nn.Tanh(),
    
            # remember to exp this
        )
        self.t_net = nn.Sequential(
            nn.Linear(input_size - input_size//2 +condition_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size//2)
        )
        self.block_det = 0
        

    def forward(self, x:torch.Tensor, y: torch.Tensor = torch.ones((0,),device=device)): # assume x is of shape (batch_size, input_size)
        x1,x2 = torch.split(x, [x.size(1) - x.size(1)//2,x.size(1)//2], dim=1)  # creates views!
        x1_cond = torch.cat([x1,y],dim=1)
        scaled_x1 = self.scale_net(x1_cond)
        z2 = x2 * torch.exp(scaled_x1) + self.t_net(x1_cond) # could be exp(a * ...), but also for backwards.
        # where a is hyperparam
        z = torch.cat([x1, z2], dim=1)
        #if self.training: 
        # # just always compute, because we need it for loss calculation
        # its not necesarry for inference, but we only do training and testing 
        self.block_det = torch.sum(scaled_x1) 
        return z
        
    def reverse(self, z, y):
        z1,z2 = torch.split(z,[z.size(1) - z.size(1)//2,z.size(1)//2], dim=1)
        x1 = torch.cat([z1,y],dim=1)
        x2 = (z2 - self.t_net(x1)) / torch.exp(self.scale_net(x1))
        x = torch.cat([z1,x2],dim=1)
        return x

class RealNVP(nn.Module):
    condition_size: int
    def __init__(self,input_size:int, hidden_size:int, blocks:int, condition_size:int = 0):
        super(RealNVP, self).__init__()
        self.input_size = input_size
        self.condition_size = condition_size
        self.blocks:nn.ModuleList[CouplingBlock] = nn.ModuleList()
        self.rotation_matrices = []
        for i in range(blocks -1):
            self.blocks.append(CouplingBlock(input_size , hidden_size, condition_size))
            R = get_orthonormal_matrix(input_size) # R means Rotational matrix, 
            # could also be named Q.
            R.requires_grad = False # prof said its not worth to learn (probably)
            # but it seams to decrease train loss. # but reconstruction is not possible if R becomes not Orthonormal
            self.rotation_matrices.append(R)

        #!TODO: readd
        self.blocks.append(CouplingBlock(input_size, hidden_size, condition_size))

    def forward(self, x, y=torch.ones((0,),device=device)):
        """
        y should be a onehot
        """
        for block, R in zip(self.blocks, self.rotation_matrices):  # pyright: ignore[reportAssignmentType]
            block:CouplingBlock
            # last from blocks will not be used in this loop
            x = block.forward(x, y)
            x = x @ R  # rotation 
        #!TODO: readd
        x = self.blocks[-1].forward(x, y)
        return x
    def reverse(self, z:torch.Tensor, y:torch.Tensor = torch.ones((0,))):
        #!TODO: readd
        #x = z
        x = self.blocks[-1].reverse(z, y)  # pyright: ignore[reportCallIssue]
        for block, R in zip(reversed( self.blocks[:-1]),reversed(self.rotation_matrices)):  # pyright: ignore[reportAssignmentType]
            block:CouplingBlock
            x = x @ R.T # TODO: batch
            x = block.reverse(x, y)
        return x

    def sample(self,num_samples:int, conditions: Optional[torch.Tensor] = None ):
        effective_num_samples = num_samples if conditions is None else num_samples * conditions.size(0)
        gaussians = torch.randn(effective_num_samples, self.input_size,device=device)
        if conditions is not None: # conditions is provided
            conditions = conditions.repeat(num_samples, 1)
        else: # conditions is not provided
            if self.condition_size == 0: # stack empty tensor
                conditions = torch.ones((num_samples,0),device=device)
            else: # is a conditional model, but no classes were supplied, so draw random
                conditions = torch.randint(0, self.condition_size, (num_samples,),device=device)
        return self.reverse(gaussians, conditions)
        

In [ ]:
# adapt train function from ex3
from torch.amp.grad_scaler import GradScaler

lam_mean = torch.tensor(0.15)
lam_std  = torch.tensor(0.05)


mu_mean = torch.tensor(0.1)
mu_std  = torch.tensor(0.025)


I0_mean = torch.tensor(I0_mean)

I0_std = torch.sqrt(I0_mean)

std_tensor = torch.tensor([lam_std,mu_std,I0_std],device=device).reshape((1,3))
mean_tensor = torch.tensor([lam_mean,mu_mean,I0_mean],device=device).reshape((1,3))
def y_scaling(y:torch.Tensor):
    # y shape: bs x 3
    return (y - mean_tensor) / std_tensor

def y_unscale(y_scaled: torch.Tensor):
    return y_scaled * std_tensor + mean_tensor

def calculate_loss(model, output):
    loss = 0
    for block in model.realNVP.blocks:
        block:CouplingBlock
        loss -= block.block_det
    loss+= output.pow(2).sum() / 2 # sum over all axis
    return loss / output.size(0)  # mean over batch

import torch.nn.utils as utils
def train_inn_cond(model:nn.Module, train_set_fn,optim, epochs, batch_size = 128):
    scaler = GradScaler("cuda", enabled=(device.type == 'cuda'))
    #train_dataloader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True,pin_memory=True) 
    
    loss_history = []
     
    for epoch in range(epochs):
        epoch_loss = 0

        # np land:
        X,Y = train_set_fn(batch_size) # ignore label
        
        # Y = sample_from_prior(batch_size)
        # X , _ = simulate_sir_simple(Y[:,0],Y[:,1],Y[:,2]) # lam, mu, i0


        # torch land:
        X = torch.tensor(X.reshape(batch_size,-1),device=device,dtype=torch.float32)   
        # Y = torch.tensor(Y,device=device,dtype=torch.float32)
        Y = y_scaling(Y)
        optim.zero_grad()

        #with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):  # pyright: ignore[reportPrivateImportUsage]
        output = model.forward(X,Y) # x is used for condition, y is input
        loss = calculate_loss(model,output)
        # scaler.scale(loss).backward()
        utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        #scaler.step(optim)
        #scaler.update()
        optim.step()
        epoch_loss += loss.item()
        avg_epoch_loss = epoch_loss
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_epoch_loss:.4f}",end="\r")
        loss_history.append(avg_epoch_loss)

    return loss_history

In [ ]:
class RealNVPSummary(nn.Module):
    def __init__(self,input_size = 200, condition_size = 8,s_hidden = 1000,s_layers=6, r_hidden = 500,r_blocks=6, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.summary = RegressionNetwork(input_size,hidden_size= s_hidden, layers=s_layers, output_size=condition_size)
        self.realNVP = RealNVP(input_size=3,hidden_size=r_hidden,blocks=r_blocks,condition_size=condition_size)

    def forward(self,x,y = None):
        """
        x is observation, y is [lam,mu,i0]
        """
        hx = self.summary(x)
        codes = self.realNVP(y,hx)
        return codes
    def reverse(self,z,x):
        hx = self.summary(x)
        y = self.realNVP.reverse(z,hx)
        return y
    
    def sample(self,x,n):
        """
        samples n per condition
        """
        hx = self.summary(x)
        return self.realNVP.sample(n,hx)

In [ ]:
lam_gen = torch.distributions.Gamma(torch.tensor(lam_k),torch.tensor(1/lam_theta)) # 1/theta to get beta
mu_gen = torch.distributions.Gamma(torch.tensor(mu_k),torch.tensor(1/mu_theta))
I0_gen = torch.distributions.Poisson(torch.tensor(I0_mean,dtype=torch.float32))


def gen_xy_torch(batch_size,T:int = 100):
    dtype = torch.float32

    Y = torch.zeros((batch_size,3),dtype=dtype,device=device)
    Y[:,0] = lam_gen.sample((batch_size,))
    Y[:,1] = mu_gen.sample((batch_size,))
    Y[:,2] = I0_gen.sample((batch_size,))



    SIR = torch.zeros((batch_size,3),device=device)
    SIR[:,0] = 900 # S
    SIR[:,1] = Y[:,2] # I
    SIR[:,2] = 10 # R
    N = SIR.sum(dim=-1)

    observations = torch.zeros((batch_size,T,2),dtype=dtype,device=device)
    observations_i = torch.zeros((batch_size,))

    for i in range(T):
        SIN = SIR[:,0] * SIR[:,1] / N
        
        observations[:,i,0] = -Y[:,0] * SIN # delta_s
        #delta_S = -lam * S * I / N
        
        observations_i = Y[:,0] * SIN - Y[:,1] * SIR[:,1]
        #delta_I = lam * S * I / N - mu * I
        
        observations[:,i,1] = Y[:,1] * SIR[:,1]
        #delta_R = mu * I



        SIR[:,0] += observations[:,i,0]
        SIR[:,1] += observations_i
        SIR[:,2] += observations[:,i,1]

        # S += delta_S
        # I += delta_I
        # R += delta_R

        

        #obs = Observation(delta_S ,delta_R)
        #observations[:,i,:] = np.array([delta_S,delta_R]).T
        # variables = SIRAmounts(S,I,R)
        #amounts[:,i,:] = np.array((S,I,R)).T
        
    return observations, Y

#### new code

In [ ]:
@torch.no_grad()
def make_calibration_test_set(N=200, T=100):
    observations, Y = gen_xy_torch(N, T)
    X = observations.reshape(N, -1)

    return X, Y


In [ ]:
import math

def log_posterior(model: RealNVPSummary, x, y):
    """
    x: (1, input_dim)
    y: (N, 3)   unscaled!
    returns: log p(y | x)  shape (N,)
    """
    model.eval()

    y_scaled = y_scaling(y)

    h_x = model.summary(x)  
    h_x = h_x.repeat(y.shape[0], 1) 

    z = model.realNVP.forward(y_scaled, h_x)

    log_pz = -0.5 * (z**2).sum(dim=1)
    log_pz -= 0.5 * z.shape[1] * math.log(2 * math.pi)

    log_det = torch.zeros_like(log_pz)
    for block in model.realNVP.blocks:
        log_det += block.block_det

    return log_pz + log_det


In [ ]:
@torch.no_grad()
def marginal_calibration_ranks(
    model,
    X,          
    Y_true,     
    n_post_samples=1000
):
    model.eval()
    N, D = Y_true.shape
    ranks = [[] for _ in range(D)]

    for i in range(N):
        x_i = X[i:i+1]
        y_star = Y_true[i]

        samples = model.sample(x_i, n_post_samples)
        samples = y_unscale(samples).cpu().numpy()

        for j in range(D):
            s_j = np.sort(samples[:, j])
            rank = np.searchsorted(s_j, y_star[j].item())
            r_ij = rank / (n_post_samples + 1)
            ranks[j].append(r_ij)

    return [np.array(r) for r in ranks]


In [ ]:
def plot_hist_and_cdf(ranks, names):
    D = len(ranks)
    fig, axes = plt.subplots(2, D, figsize=(4*D, 6))

    for j in range(D):
        # Histogram
        axes[0, j].hist(ranks[j], bins=10, density=True)
        axes[0, j].axhline(1.0, color="red", linestyle="--")
        axes[0, j].set_title(f"{names[j]} - histogram")

        # Empirical CDF
        r = np.sort(ranks[j])
        cdf = np.arange(1, len(r)+1) / len(r)
        axes[1, j].plot(r, cdf)
        axes[1, j].plot([0,1], [0,1], "--")
        axes[1, j].set_title(f"{names[j]} - CDF")

    plt.tight_layout()
    plt.show()


In [ ]:
train_history = []
model_rnvp = RealNVPSummary(input_size=200,condition_size=8,r_blocks=16,r_hidden=256,s_hidden=784,s_layers=4).to(device)
lr = 1e-3

try:
    from lion_pytorch import Lion
    from lion_pytorch.cautious_lion import Lion as CLion
    optim = CLion(model_rnvp.parameters(),lr)
except Exception:
    pass
optim = torch.optim.Adam(model_rnvp.parameters(),lr)
train_losses = train_inn_cond(model_rnvp,gen_xy_torch,optim,epochs=1000,batch_size=128)
train_history.extend(train_losses)

In [ ]:
# lower lr of optim by 10:
for param_group in optim.param_groups:
    param_group["lr"] /= 10

train_losses = train_inn_cond(model_rnvp,gen_xy_torch,optim,epochs=1000,batch_size=64)
train_history.extend(train_losses)
plt.plot(train_history)
plt.title("Training loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")

#### Marginal calibration

In [ ]:
X_cal, Y_cal = make_calibration_test_set(N=200)

ranks = marginal_calibration_ranks(
    model_rnvp,
    X_cal,
    Y_cal,
    n_post_samples=1000
)

plot_hist_and_cdf(
    ranks,
    names=[r"$\lambda$", r"$\mu$", r"$I_0$"]
)


cdf looks pretty good, histograms could be better.

#### Joint calibration via surprisal

In [ ]:
@torch.no_grad()
def compute_surprisal(model, x_cond, y_samples):
    """
    On-the-fly computation of -log p(y|x) without modifying the model.
    """
    y_scaled = y_scaling(y_samples)
    N = y_samples.shape[0]
    h_x = model.summary(x_cond)
    h_x = h_x.repeat(N, 1)

    z = y_scaled
    total_log_det = torch.zeros(N, device=y_samples.device)

    # all blocks except last
    for block, R in zip(model.realNVP.blocks[:-1], model.realNVP.rotation_matrices):
        x1, x2 = torch.split(z, [z.size(1) - z.size(1)//2, z.size(1)//2], dim=1)
        x1_cond = torch.cat([x1, h_x], dim=1)
        s = block.scale_net(x1_cond)
        t = block.t_net(x1_cond)
        z2 = x2 * torch.exp(s) + t
        z = torch.cat([x1, z2], dim=1)
        total_log_det += s.sum(dim=1)
        z = z @ R

    # last block
    block = model.realNVP.blocks[-1]
    x1, x2 = torch.split(z, [z.size(1) - z.size(1)//2, z.size(1)//2], dim=1)
    x1_cond = torch.cat([x1, h_x], dim=1)
    s = block.scale_net(x1_cond)
    t = block.t_net(x1_cond)
    z2 = x2 * torch.exp(s) + t
    z = torch.cat([x1, z2], dim=1)
    total_log_det += s.sum(dim=1)

    log_pz = -0.5 * (z ** 2).sum(dim=1)
    log_pz -= 0.5 * z.size(1) * math.log(2 * math.pi)

    return -(log_pz + total_log_det)

In [ ]:
@torch.no_grad()
def surprisal_ranks(model, X, Y_true, n_post_samples=1000):
    """
    Computes rank statistics for joint surprisal E(Y) = -log p(Y|X)
    """
    N = Y_true.shape[0]
    ranks = []
    model.eval()

    for i in range(N):
        x_i = X[i:i+1]
        y_star = Y_true[i:i+1]

        Y_samples = model.sample(x_i, n_post_samples)
        Y_samples = y_unscale(Y_samples)#.cpu()
        y_star_cpu = y_star#.cpu()

        E_samples = compute_surprisal(model, x_i, Y_samples)
        E_true = compute_surprisal(model, x_i, y_star_cpu)

        sorted_E = np.sort(E_samples.cpu().numpy())
        rank = np.searchsorted(sorted_E, E_true.cpu().item())
        r = rank / (n_post_samples + 1)
        ranks.append(r)

    return np.array(ranks)


In [ ]:
def plot_calibration_histograms(ranks_list, bins=10, titles=None):
    D = len(ranks_list)
    fig, axes = plt.subplots(1, D, figsize=(4*D, 3))
    if D == 1: axes = [axes]

    for j in range(D):
        axes[j].hist(ranks_list[j], bins=bins, density=True)
        axes[j].axhline(1.0, color="red", linestyle="--")
        axes[j].set_title(titles[j] if titles else f"Calibration dim {j}")

    plt.tight_layout()
    plt.show()

def plot_surprisal_cdf(ranks):
    r_sorted = np.sort(ranks)
    cdf = np.arange(1, len(r_sorted)+1) / len(r_sorted)
    plt.plot(r_sorted, cdf)
    plt.plot([0,1],[0,1],"--", color="red")
    plt.title("Surprisal CDF")
    plt.show()


In [ ]:
supr_ranks = surprisal_ranks(model_rnvp, X_cal, Y_cal, n_post_samples=1000)
plt.hist(supr_ranks, bins=10, density=True)
plt.axhline(1.0, color="red", linestyle="--")
plt.title("Joint Surprisal Histogram")
plt.show()
plot_surprisal_cdf(supr_ranks)


# TASK 2:

We copy all the code from the previous exercise and only change the simulation.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gamma, poisson

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

In [ ]:
import numpy as np

def simulate_sir_noisy(lam, mu, I_0, L, rho, sigma_2, T: int = 100,return_amounts = True):
    dtype = np.float32

    #convert to arrays
    lam     = np.asarray(lam,     dtype=dtype)
    mu      = np.asarray(mu,      dtype=dtype)
    I       = np.asarray(I_0,     dtype=dtype)
    rho     = np.asarray(rho,     dtype=dtype)
    sigma_2 = np.asarray(sigma_2, dtype=dtype)
    L       = np.asarray(L)

    
    if lam.ndim == 0:
        lam     = lam[None]
        mu      = mu[None]
        I       = I[None]
        rho     = rho[None]
        sigma_2 = sigma_2[None]
        L       = L[None]

    batch_size = lam.shape[0]

    #Shape-Checks
    assert mu.shape      == (batch_size,)
    assert I.shape       == (batch_size,)
    assert rho.shape     == (batch_size,)
    assert sigma_2.shape == (batch_size,)

    # L is constant
    if L.ndim == 0:
        L_int = int(L)
    else:
        if not np.all(L == L[0]):
            raise ValueError("L must be the same for all samples (constant).")
        L_int = int(L[0])

    if L_int < 0:
        raise ValueError("L must be >= 0")
    if np.any(sigma_2 < 0):
        raise ValueError("sigma_2 must be >= 0")

    
    S = np.full(batch_size, 900, dtype=dtype)
    R = np.full(batch_size, 10,  dtype=dtype)
    N = S + I + R  

    S0 = S.copy()
    I0 = I.copy()
    R0 = R.copy()


    observations = np.zeros((batch_size, T + L_int, 2), dtype=dtype)  # (ΔS, ΔR)
    amounts      = np.zeros((batch_size, T + L_int, 3), dtype=dtype)  # (S, I, R)

    # SIR-Simulation without noise
    for t in range(T + L_int):
        delta_S = -lam * S * I / N
        delta_I =  lam * S * I / N - mu * I
        delta_R =  mu * I

        S += delta_S
        I += delta_I
        R += delta_R

        observations[:, t, :] = np.stack([delta_S, delta_R], axis=-1)
        amounts[:, t, :]      = np.stack([S, I, R], axis=-1)

    noisy_observations = observations[:, L_int:L_int + T, :] 

    
    # changed to sample new noise for every timestep
    # changed to only resample invalid (<0) noise factors
    eps = None
    candidate = np.random.normal(loc=rho[:,None, None],scale=np.sqrt(sigma_2)[:,None, None], size=noisy_observations.shape).astype(dtype).flatten()
    is_invalid = (candidate < 0).flatten()
    while is_invalid.sum() > 0:
        replace_candidates = np.random.normal(loc=rho[:,None, None],scale=np.sqrt(sigma_2)[:,None, None], size=noisy_observations.shape).astype(dtype).flatten()
        candidate[is_invalid] = replace_candidates[is_invalid]
        
        is_invalid = (candidate < 0)
    eps = candidate.reshape(noisy_observations.shape)
    
    noisy_observations *= eps

    #reconstruct amounts from observations
    
    if return_amounts:

        if L_int > 0:
            S_base = amounts[:, L_int - 1, 0]
            # I_base = amounts[:, L_int - 1, 1]
            R_base = amounts[:, L_int - 1, 2]
        else:
            S_base = S0
            # I_base = I0
            R_base = R0

        delta_S_noisy = noisy_observations[:, :, 0]  # (batch, T)
        delta_R_noisy = noisy_observations[:, :, 1]

        S_noisy = S_base[:, None] + np.cumsum(delta_S_noisy, axis=1)
        R_noisy = R_base[:, None] + np.cumsum(delta_R_noisy, axis=1)
        I_noisy = N[:, None] - S_noisy - R_noisy

        noisy_amounts = np.stack([S_noisy, I_noisy, R_noisy], axis=-1).astype(dtype)

        return noisy_observations, noisy_amounts, observations, amounts
    return noisy_observations


#later when using this simulation to train the networks, we noticed this simulation is extremely slow, so now we implement a light version:

def simulate_sir_simple_noisy_light(lam, mu, I_0, L, rho, sigma_2, T: int = 100):
    dtype = np.float32

    lam     = np.asarray(lam,     dtype=dtype)
    mu      = np.asarray(mu,      dtype=dtype)
    I       = np.asarray(I_0,     dtype=dtype)
    L       = np.asarray(L)
    rho     = np.asarray(rho,     dtype=dtype)
    sigma_2 = np.asarray(sigma_2, dtype=dtype)

    if lam.ndim == 0:
        lam     = lam[None]
        mu      = mu[None]
        I       = I[None]
        L       = L[None]
        rho     = rho[None]
        sigma_2 = sigma_2[None]

    batch_size = lam.shape[0]

    if L.ndim == 0:
        L_int = int(L)
    else:
        L_int = int(L[0])

    S = np.full(batch_size, 900, dtype=dtype)
    R = np.full(batch_size, 10,  dtype=dtype)
    N = S + I + R

    observations = np.zeros((batch_size, T + L_int, 2), dtype=dtype)

    for t in range(T + L_int):
        delta_S = -lam * S * I / N
        delta_I =  lam * S * I / N - mu * I
        delta_R =  mu * I

        S += delta_S
        I += delta_I
        R += delta_R

        observations[:, t, :] = np.stack([delta_S, delta_R], axis=-1)

    noisy_observations = observations[:, L_int:L_int + T, :].copy()

    if np.any(sigma_2 < 0):
        raise ValueError("sigma_2 must be >= 0")

    if rho.ndim == 0:
        rho = np.full(batch_size, rho, dtype=dtype)
    if sigma_2.ndim == 0:
        sigma_2 = np.full(batch_size, sigma_2, dtype=dtype)

    eps = np.random.normal(
        loc=rho[:, None],
        scale=np.sqrt(sigma_2)[:, None],
        size=(batch_size, 2)
    ).astype(dtype)
    eps[eps < 0] = 0.0

    noisy_observations *= eps[:, None, :]

    return noisy_observations



In [ ]:
noisy_observations, noisy_amounts, observations, amounts = simulate_sir_noisy(lam=np.array(0.15),mu=np.array(0.02),I_0=np.array(9), L=0, rho = np.array(1), sigma_2=np.array(0.5),T=100,return_amounts=True)
plt.plot(noisy_amounts[0,:,0])
plt.plot(amounts[0,:,0])


Evalution of "good" sigmas

In [ ]:


sigma_values = np.linspace(0, 1, 10)   # 10 sigma² values 0..1
L_values = np.arange(1, 11)            # L = 1..10
T = 100

lam = np.array(0.15)
mu  = np.array(0.1)
I0  = np.array(15)
rho = np.array(1)

num_simulations = 5   # how many full runs

for sim in range(1, num_simulations + 1):
    fig, axes = plt.subplots(len(sigma_values), len(L_values),
                             figsize=(22, 18), sharex=True, sharey=True)
    axes = axes.reshape(len(sigma_values), len(L_values))

    for row, sigma_2 in enumerate(sigma_values):
        for col, L in enumerate(L_values):
            ax = axes[row, col]

            # true curve (only once per row)
            if col == 0:
                _, _, _, true_amounts = simulate_sir_noisy(
                    lam=lam, mu=mu, I_0=I0, L=0, rho=rho,
                    sigma_2=np.array(sigma_2), T=T
                )
                true_S = true_amounts[0, :, 0]

            noisy_obs, noisy_amounts, _, _ = simulate_sir_noisy(
                lam=lam, mu=mu, I_0=I0, L=L, rho=rho,
                sigma_2=np.array(sigma_2), T=T
            )
            noisy_S = noisy_amounts[0, :, 0]

            ax.plot(true_S, color="black", linewidth=1.5, label="True S(t)")
            ax.plot(noisy_S, color="tab:blue", alpha=0.85, label="Noisy S(t)")

            if col == 0:
                ax.set_ylabel(f"σ² = {sigma_2:.2f}", fontsize=9)

            if row == 0:
                ax.set_title(f"L = {L}", fontsize=10)

    # extract legend only once
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right",
               bbox_to_anchor=(0.99, 0.99), fontsize=10)

    # shared axis labels
    fig.text(0.5, 0.04, "Time (t)", ha="center", fontsize=13)
    fig.text(0.04, 0.5, "S(t)", va="center", rotation="vertical", fontsize=13)

    # --- title of full grid ---
    fig.suptitle(f"Simulation {sim} — True vs. Noisy S(t) "
                 f"for all (σ², L) combinations",
                 fontsize=16)

    plt.tight_layout(rect=[0, 0.05, 0.96, 0.96])
    plt.show()


Comment: Interesting data seems to occur at L=10 and sigma_2 between 0.5 and 0.7. These the real and the noisy data is not identical or just shifted in the y axis (like in many other combinations), so we will use L=10 and uniform sigma_2 between 0.5 and 0.7 in the new prior. Changing rho to me does not make that much sense, because it does not really change the behaviour of the simulation if we equally diminish all observed values.

Extending the prior with the values for rho, sigma, L we found before (rho = 1; sigma = 0.6; L = 10)

In [ ]:
#new prior


lam_mean = 0.15
lam_std  = 0.05


mu_mean = 0.1
mu_std  = 0.025


I0_mean = 15

L = 10
sigma_lower = 0.5
sigma_higher = 0.7

n_samples = 1000



def gamma_from_mean_std(mean, std):
    shape = (mean / std) ** 2    
    scale = (std ** 2) / mean     
    return shape, scale

 
lam_k, lam_theta = gamma_from_mean_std(lam_mean, lam_std)
mu_k,  mu_theta  = gamma_from_mean_std(mu_mean,  mu_std)

  

def sample_from_prior(size):
    lam_samples = gamma.rvs(a=lam_k, scale=lam_theta, size=size)
    mu_samples  = gamma.rvs(a=mu_k,  scale=mu_theta,  size=size)
    I0_samples  = poisson.rvs(mu=I0_mean, size=size)


    I0_samples = np.maximum(I0_samples, 1)

    sigma_2_samples = np.random.uniform(low=sigma_lower, high=sigma_higher, size=size)
    L_samples = np.ones_like(sigma_2_samples) * 10
    rho_samples = np.ones_like(sigma_2_samples)


    return np.column_stack([lam_samples, mu_samples, I0_samples, L_samples, rho_samples, sigma_2_samples])



Testing our new simulation and prior

In [ ]:
batch_size = 3

prior_sample = sample_from_prior(batch_size) 
print(prior_sample.shape)
noisy_observations, noisy_amounts, observations, amounts = simulate_sir_noisy(lam=prior_sample[:,0],mu=prior_sample[:,1],I_0=prior_sample[:,2], L =prior_sample[:,3],rho =prior_sample[:,4],sigma_2 = prior_sample[:,5] ,T=100)
# for i in range(observations.shape[0]):
#     plt.plot(observations[i])

# plt.ylim(-20,20)

for i in range(batch_size):
    plt.plot(noisy_observations[i,:,0], label=f"ΔS sim {i+1}")
    plt.plot(noisy_observations[i,:,1], label=f"ΔR sim {i+1}")

plt.legend()
plt.ylim(-20,20)
plt.show()

Regression Network for point estimation:


In [ ]:
# normalize predicted variables ():

    # we could do this empirically with calculating from:
    # prior_samples = sample_from_prior(10_000)

lam_mean = torch.tensor(0.15)
lam_std  = torch.tensor(0.05)


mu_mean = torch.tensor(0.1)
mu_std  = torch.tensor(0.025)

I0_mean = 15
I0_mean = torch.tensor(I0_mean)

I0_std = torch.sqrt(I0_mean)

std_tensor = torch.tensor([lam_std,mu_std,I0_std],device=device).reshape((1,3))
mean_tensor = torch.tensor([lam_mean,mu_mean,I0_mean],device=device).reshape((1,3))
# 1 x 3

def y_rescale(y_normalized:torch.Tensor):
    return y_normalized * std_tensor + mean_tensor

def y_scaling(y):
    # y shape: bs x 3
    return (y - mean_tensor) / std_tensor

class RegressionNetwork(nn.Module):
    def __init__(self, input_size=200, hidden_size=400, layers=6, output_size = 3):
        """
        
        """
        super().__init__()
        self.input_size = input_size
        self.layers = layers
        self.encoder_layers = []
        self.encoder_layers.append((nn.Linear(input_size,hidden_size)))
        self.encoder_layers.append(nn.ReLU())
        for i in range(layers):
            self.encoder_layers.append(nn.Linear(hidden_size,hidden_size))
            self.encoder_layers.append(nn.ReLU())
        self.encoder_layers.append(nn.Linear(hidden_size,output_size))

        self.encoder = nn.Sequential(*self.encoder_layers)

    def forward(self,x):
        return self.encoder(x)
    
def train_regression_network_light(model, batch_size=64, T=100, epochs=500, learning_rate=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    train_losses = []

    for epoch in range(epochs):
        prior = sample_from_prior(batch_size)
        lam     = prior[:, 0]
        mu      = prior[:, 1]
        I0      = prior[:, 2]
        L       = prior[:, 3]
        rho     = prior[:, 4]
        sigma_2 = prior[:, 5]

        noisy_obs = simulate_sir_simple_noisy_light(
            lam=lam, mu=mu, I_0=I0, L=L, rho=rho, sigma_2=sigma_2, T=T
        )

        X = noisy_obs.reshape(batch_size, -1)
        y = np.column_stack([lam, mu, I0])

        X_torch = torch.tensor(X, dtype=torch.float32, device=device)
        y_torch = torch.tensor(y, dtype=torch.float32, device=device)

        y_torch = y_scaling(y_torch)

        pred = model(X_torch)
        loss = criterion(pred, y_torch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    return train_losses

def train_regression_network(model, batch_size=64, T=100, epochs=500, learning_rate=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    train_losses = []

    for epoch in range(epochs):
        prior = sample_from_prior(batch_size)
        lam     = prior[:, 0]
        mu      = prior[:, 1]
        I0      = prior[:, 2]
        L       = prior[:, 3]
        rho     = prior[:, 4]
        sigma_2 = prior[:, 5]

        noisy_obs = simulate_sir_noisy(
            lam=lam, mu=mu, I_0=I0, L=L, rho=rho, sigma_2=sigma_2, T=T,return_amounts=False
        )

        X = noisy_obs.reshape(batch_size, -1)
        y = np.column_stack([lam, mu, I0])

        X_torch = torch.tensor(X, dtype=torch.float32, device=device)
        y_torch = torch.tensor(y, dtype=torch.float32, device=device)

        y_torch = y_scaling(y_torch)

        pred = model(X_torch)
        loss = criterion(pred, y_torch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    return train_losses



def regression_model_test_new(model, samples=10, T=100):
    predicted_params = []
    true_params = []

    for _ in range(samples):
        prior = sample_from_prior(1)[0]
        lam, mu, I0, L, rho, sigma2 = prior
        true_params.append([lam, mu, I0])

        noisy_obs, _, _, _ = simulate_sir_noisy(
            lam=np.array([lam]),
            mu=np.array([mu]),
            I_0=np.array([I0]),
            L=np.array([L]),
            rho=np.array([rho]),
            sigma_2=np.array([sigma2]),
            T=T
        )

        X_test = noisy_obs.reshape(1, -1)
        X_test_torch = torch.tensor(X_test, dtype=torch.float32, device=device)

        pred = model(X_test_torch)
        pred = y_rescale(pred)

        predicted_params.append(pred.detach().cpu().numpy())

    predicted_params = np.array(predicted_params).squeeze()
    true_params = np.array(true_params)
    return predicted_params, true_params



def plot_regressiom_model_test_new(model, samples=10, T=100):
    predicted_params, true_params = regression_model_test_new(model, samples, T)

    pred_array = np.array(predicted_params)
    true_array = np.array(true_params)

    param_names = [r'$\lambda$', r'$\mu$', r'$I_0$']
    x = range(len(true_array))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for i in range(3):
        axes[i].plot(x, true_array[:, i], 'o-', label='True', markersize=6)
        axes[i].plot(x, pred_array[:, i], 's--', label='Predicted', markersize=6)
        axes[i].set_title(f'Parameter: {param_names[i]}')
        axes[i].set_xlabel('Sample Index')
        axes[i].set_ylabel('Value')
        axes[i].legend()
        axes[i].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    criterion = nn.MSELoss(reduction='none')
    loss = criterion(torch.tensor(pred_array), torch.tensor(true_array))

    plt.figure()
    plt.plot(loss[:, 0].detach().cpu().numpy(), label="lam")
    plt.plot(loss[:, 1].detach().cpu().numpy(), label="mu")
    plt.plot(loss[:, 2].detach().cpu().numpy(), label="I0")
    plt.yscale("log")
    plt.title("MSE of each parameter and simulation example.")
    plt.legend()
    plt.show()





In [ ]:
print(device)
model3 = RegressionNetwork(input_size=200,hidden_size=800,layers=5)
model3 = model3.to(device)
train_losses = train_regression_network(model3, epochs=5000,learning_rate=1e-4)
import matplotlib.pyplot as plt

plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss Over Epochs')
plt.show()
plot_regressiom_model_test_new(model3,10)

Comment: We observe a very similar behaviour to the simulation without noise: with enough training, the regression network does a very good job at estimating the hidden params, even from noisy data.

We now will repeat the evaluation of the INN

In [ ]:
# task 3:
# ex 3 copy of realnvp

# define RealNVP, and its building blocks
# device that is either cpu, cuda or mps
from typing import Optional

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")
#device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

def to_onehot(y,num_classes:int = 2):
    return torch.nn.functional.one_hot(y, num_classes=num_classes)


def get_orthonormal_matrix(d:int) -> torch.Tensor:
    A = torch.randn((d,d))
    Q, _R = torch.linalg.qr(A)
    return Q.to(device)

class CouplingBlock(nn.Module):
    def __init__(self, input_size:int, hidden_size:int, condition_size:int = 0):
        super(CouplingBlock, self).__init__()
        self.scale_net = nn.Sequential(
            nn.Linear(input_size - input_size//2 + condition_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size//2),
            nn.Tanh(),
    
            # remember to exp this
        )
        self.t_net = nn.Sequential(
            nn.Linear(input_size - input_size//2 +condition_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size//2)
        )
        self.block_det = 0
        

    def forward(self, x:torch.Tensor, y: torch.Tensor = torch.ones((0,),device=device)): # assume x is of shape (batch_size, input_size)
        x1,x2 = torch.split(x, [x.size(1) - x.size(1)//2,x.size(1)//2], dim=1)  # creates views!
        x1_cond = torch.cat([x1,y],dim=1)
        scaled_x1 = self.scale_net(x1_cond)
        z2 = x2 * torch.exp(scaled_x1) + self.t_net(x1_cond) # could be exp(a * ...), but also for backwards.
        # where a is hyperparam
        z = torch.cat([x1, z2], dim=1)
        #if self.training: 
        # # just always compute, because we need it for loss calculation
        # its not necesarry for inference, but we only do training and testing 
        self.block_det = torch.sum(scaled_x1) 
        return z
        
    def reverse(self, z, y):
        z1,z2 = torch.split(z,[z.size(1) - z.size(1)//2,z.size(1)//2], dim=1)
        x1 = torch.cat([z1,y],dim=1)
        x2 = (z2 - self.t_net(x1)) / torch.exp(self.scale_net(x1))
        x = torch.cat([z1,x2],dim=1)
        return x

class RealNVP(nn.Module):
    condition_size: int
    def __init__(self,input_size:int, hidden_size:int, blocks:int, condition_size:int = 0):
        super(RealNVP, self).__init__()
        self.input_size = input_size
        self.condition_size = condition_size
        self.blocks:nn.ModuleList[CouplingBlock] = nn.ModuleList()
        self.rotation_matrices = []
        for i in range(blocks -1):
            self.blocks.append(CouplingBlock(input_size , hidden_size, condition_size))
            R = get_orthonormal_matrix(input_size) # R means Rotational matrix, 
            # could also be named Q.
            R.requires_grad = False # prof said its not worth to learn (probably)
            # but it seams to decrease train loss. # but reconstruction is not possible if R becomes not Orthonormal
            self.rotation_matrices.append(R)

        #!TODO: readd
        self.blocks.append(CouplingBlock(input_size, hidden_size, condition_size))

    def forward(self, x, y=torch.ones((0,),device=device)):
        """
        y should be a onehot
        """
        for block, R in zip(self.blocks, self.rotation_matrices):  # pyright: ignore[reportAssignmentType]
            block:CouplingBlock
            # last from blocks will not be used in this loop
            x = block.forward(x, y)
            x = x @ R  # rotation 
        #!TODO: readd
        x = self.blocks[-1].forward(x, y)
        return x
    def reverse(self, z:torch.Tensor, y:torch.Tensor = torch.ones((0,))):
        #!TODO: readd
        #x = z
        x = self.blocks[-1].reverse(z, y)  # pyright: ignore[reportCallIssue]
        for block, R in zip(reversed( self.blocks[:-1]),reversed(self.rotation_matrices)):  # pyright: ignore[reportAssignmentType]
            block:CouplingBlock
            x = x @ R.T # TODO: batch
            x = block.reverse(x, y)
        return x

    def sample(self,num_samples:int, conditions: Optional[torch.Tensor] = None ):
        effective_num_samples = num_samples if conditions is None else num_samples * conditions.size(0)
        gaussians = torch.randn(effective_num_samples, self.input_size,device=device)
        if conditions is not None: # conditions is provided
            conditions = conditions.repeat(num_samples, 1)
        else: # conditions is not provided
            if self.condition_size == 0: # stack empty tensor
                conditions = torch.ones((num_samples,0),device=device)
            else: # is a conditional model, but no classes were supplied, so draw random
                conditions = torch.randint(0, self.condition_size, (num_samples,),device=device)
        return self.reverse(gaussians, conditions)
        
# adapt train function from ex3
from torch.amp.grad_scaler import GradScaler

def calculate_loss(model, output):
    loss = 0
    for block in model.realNVP.blocks:
        block:CouplingBlock
        loss -= block.block_det
    loss+= output.pow(2).sum() / 2 # sum over all axis
    return loss / output.size(0)  # mean over batch

import torch.nn.utils as utils
from tqdm.notebook import tqdm
def train_inn_cond(model:nn.Module, train_set_fn,optim, epochs, batch_size = 128):
    scaler = GradScaler("cuda", enabled=(device.type == 'cuda'))
    #train_dataloader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True,pin_memory=True) 
    
    loss_history = []
    
    for epoch in tqdm(range(epochs)):
        epoch_loss = 0

        # np land:
        X,Y = train_set_fn(batch_size) # ignore label
        
        # Y = sample_from_prior(batch_size)
        # X , _ = simulate_sir_simple(Y[:,0],Y[:,1],Y[:,2]) # lam, mu, i0


        # torch land:
        # X = torch.tensor(X.reshape(batch_size,-1),device=device,dtype=torch.float32)   
        # assume X is already tensor on device
        # Y = torch.tensor(Y,device=device,dtype=torch.float32)
        X = X.to(device)
        Y = Y.to(device)
        Y = y_scaling(Y)
        optim.zero_grad()

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):  # pyright: ignore[reportPrivateImportUsage]
            output = model.forward(X,Y) # x is used for condition, y is input
            loss = calculate_loss(model,output)
        scaler.scale(loss).backward()
        # utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optim)
        scaler.update()
        epoch_loss += loss.item()
        avg_epoch_loss = epoch_loss
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_epoch_loss:.4f}",end="\r")
        loss_history.append(avg_epoch_loss)

    return loss_history
class RealNVPSummary(nn.Module):
    def __init__(self,input_size = 200, condition_size = 8,s_hidden = 1000,s_layers=6, r_hidden = 500,r_blocks=6, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.summary = RegressionNetwork(input_size,hidden_size= s_hidden, layers=s_layers, output_size=condition_size)
        self.realNVP = RealNVP(input_size=3,hidden_size=r_hidden,blocks=r_blocks,condition_size=condition_size)

    def forward(self,x,y = None):
        """
        x is observation, y is [lam,mu,i0]
        """
        hx = self.summary(x)
        codes = self.realNVP(y,hx)
        return codes
    def reverse(self,z,x):
        hx = self.summary(x)
        y = self.realNVP.reverse(z,hx)
        return y
    
    def sample(self,x,n):
        """
        samples n per condition
        """
        hx = self.summary(x)
        return self.realNVP.sample(n,hx)

simulate_sir_noisy_torch = torch.compile(simulate_sir_noisy)

@torch.compile
def gen_xy_torch(batch_size: int):
        prior = sample_from_prior(batch_size)
        #prior = torch.tensor( sample_from_prior(batch_size),device=device)  

        lam     = prior[:, 0]
        mu      = prior[:, 1]
        I0      = prior[:, 2]
        L       = prior[:, 3]
        rho     = prior[:, 4]
        sigma_2 = prior[:, 5]

        #with device:    
        #    X_np = simulate_sir_noisy_torch(
        X_np = simulate_sir_noisy(
            lam=lam,
            mu=mu,
            I_0=I0,
            L=L,
            rho=rho,
            sigma_2=sigma_2,
            T=T,
            return_amounts = False
            
        ) 
            
        
        X = torch.tensor(X_np, dtype=torch.float32, device=device).reshape(batch_size, -1)
        Y = torch.tensor(np.column_stack([lam, mu, I0]), dtype=torch.float32, device=device)
        return X, Y





In [ ]:
rain_history = []
model_rnvp = RealNVPSummary(input_size=200,condition_size=8,r_blocks=16,r_hidden=256,s_hidden=784,s_layers=4).to(device)
lr = 1e-3

try:
    from lion_pytorch import Lion
    from lion_pytorch.cautious_lion import Lion as CLion
    optim = CLion(model_rnvp.parameters(),lr)
except Exception:
    pass
optim = torch.optim.Adam(model_rnvp.parameters(),lr)

#NOTE: using torch compiled numpy function could be done, but i dont wanna wast the time
#train_compile = torch.compile(train_inn_cond)
#with device: 
train_losses = train_inn_cond(model_rnvp,gen_xy_torch,optim,epochs=800,batch_size=5000)
# this uses gen_xy_torch, which uses noisy simulation

In [ ]:
# just testing if reverse works:
n = 30
X,Y = gen_xy_torch(n)
X = X.reshape((n,-1))
Ys = y_scaling(Y)
gaussians = torch.randn((n,3),device=device)
gen_new_ys = model_rnvp.reverse(gaussians,X)

In [ ]:
# marginal for lam

for i in range(12):

    n = 10000
    single_y = Y[i].reshape(1,-1)
    # gaussians = torch.randn((100,3),device=device)
    single_x = X[i].reshape(1,-1)
    print(single_y.shape,single_x.shape)
    samples = model_rnvp.sample(single_x,n)
    samples = y_rescale(samples)
    print(samples.shape)

    # True:
    plt.axvline(x=single_y[0,0].detach().cpu().numpy(),color="red",label = "True")

    # Posterior
    plt.hist(samples[:,0].detach().cpu().numpy(),bins=40,density=True,label="Posterior")

    #point estimate:
    point_estimate = y_rescale( model3.forward(single_x)) [:,0]
    plt.axvline(x=point_estimate.detach().cpu().numpy(),color="black",linestyle = "--",linewidth =2,label = "Point Estimate (MLP)")



    # Plot gamma lam prior
    import scipy.stats as stats
    lam_values = torch.linspace(0, 0.5, 1000)
    gamma_prior = stats.gamma.pdf(lam_values.numpy(), a=lam_k, scale=lam_theta)  # Adjust shape and scale parameters as needed
    #plt.figure()
    plt.plot(lam_values.numpy() , gamma_prior,label = "prior" )
    plt.xlabel('Lambda')
    plt.ylabel('Density')
    plt.title('Gamma Prior for Lambda')
    plt.legend()
    plt.show()


In [ ]:
# marginal for mu

for i in range(12):
    n = 10000
    single_y = Y[i].reshape(1,-1)
    # gaussians = torch.randn((100,3),device=device)
    single_x = X[i].reshape(1,-1)
    print(single_y.shape,single_x.shape)
    samples = model_rnvp.sample(single_x,n)
    samples = y_rescale(samples)
    print(samples.shape)
    plt.axvline(x=single_y[0,1].detach().cpu().numpy(),color="red")

    print(single_y[0,1])
    plt.hist(samples[:,1].detach().cpu().numpy(),bins=40,density=True)


    #point estimate:
    point_estimate = y_rescale( model3.forward(single_x)) [:,1]
    plt.axvline(x=point_estimate.detach().cpu().numpy(),color="black",linestyle = "--",linewidth =2,label = "Point Estimate (MLP)")


    # Plot gamma mu prior
    import scipy.stats as stats
    lam_values = torch.linspace(0, 0.5, 1000)
    gamma_prior = stats.gamma.pdf(lam_values.numpy(), a=mu_k, scale=mu_theta)  # Adjust shape and scale parameters as needed
    #plt.figure()
    plt.plot(lam_values.numpy() , gamma_prior )
    plt.xlabel('Mu')
    plt.ylabel('Density')
    plt.title('Gamma Prior for Mu')
    plt.legend()
    plt.show()


In [ ]:
# marginal for I0

for i in range(12):
    n = 10000
    single_y = Y[i].reshape(1,-1)
    # gaussians = torch.randn((100,3),device=device)
    single_x = X[i].reshape(1,-1)
    print(single_y.shape,single_x.shape)
    samples = model_rnvp.sample(single_x,n)
    samples = y_rescale(samples)
    print(samples.shape)
    plt.axvline(x=single_y[0,2].detach().cpu().numpy(),color="red",label = "True")

    print(single_y[0,2])
    plt.hist(samples[:,2].detach().cpu().numpy(),bins=40,density=True,label="Posterior")

    #point estimate:
    point_estimate = y_rescale( model3.forward(single_x)) [:,2]
    plt.axvline(x=point_estimate.detach().cpu().numpy(),color="black",linestyle = "--",linewidth =2,label = "Point Estimate (MLP)")



    # Plot prior
    import scipy.stats as stats
    lam_values = torch.arange(50)
    gamma_prior = stats.poisson.pmf(lam_values.numpy(),mu=I0_mean)  # Adjust shape and scale parameters as needed
    #plt.figure()
    plt.plot(lam_values.numpy() , gamma_prior ,label="prior")
    plt.xlabel('I0')
    plt.ylabel('Density')
    plt.title('Poisson Prior for I0')
    plt.legend()
    plt.show()


Comment: We see, that the posterior for mu and lambda is very sharp in all the simulation even with noise. For I_0 the posterior sometimes is less sharp than mu and lambda, but this is due to the rescaling in the training of the network, which equalizes the relative errors and thus makes the absolute estimation for (the bigger) I_0 worse. With respect to the simulation without noise, we dont really see big differences.

In [ ]:
# heatmap:
n = 12
X,Y = gen_xy_torch(n)
X = X.reshape((n,-1))
Ys = y_scaling(Y)
gaussians = torch.randn((n,3),device=device)
gen_new_ys = model_rnvp.reverse(gaussians,X)

for i in range(12):
    n_samples = 10000
    single_y = Y[i].reshape(1,-1)
    # gaussians = torch.randn((100,3),device=device)
    single_x = X[i].reshape(1,-1)
    print(single_y.shape,single_x.shape)
    samples = model_rnvp.sample(single_x,n_samples)
    samples = y_rescale(samples).detach().cpu().numpy()
    #print(samples.shape)
    
    plt.hist2d(samples[:,0],samples[:,1],bins=40)
    plt.xlabel("lam")
    plt.ylabel("mu")
    #point estimate:
    point_estimate = y_rescale( model3.forward(single_x)) 
    point_estimate = point_estimate.detach().cpu().numpy().flatten()
    plt.scatter(point_estimate[0],point_estimate[1],label="Point Estimate",color="black")
    single_y = single_y.detach().cpu().numpy()
    plt.scatter(single_y[0,0],single_y[0,1],label="True lam, mu",color="red")
    plt.legend()
    plt.show()

In [ ]:
# heatmap for pair lambda, I_0:
n = 12
X,Y = gen_xy_torch(n)
X = X.reshape((n,-1))
Ys = y_scaling(Y)
gaussians = torch.randn((n,3),device=device)
gen_new_ys = model_rnvp.reverse(gaussians,X)

for i in range(12):
    n_samples = 10000
    single_y = Y[i].reshape(1,-1)
    # gaussians = torch.randn((100,3),device=device)
    single_x = X[i].reshape(1,-1)
    print(single_y.shape,single_x.shape)
    samples = model_rnvp.sample(single_x,n_samples)
    samples = y_rescale(samples).detach().cpu().numpy()
    #print(samples.shape)
    
    plt.hist2d(samples[:,0],samples[:,2],bins=40)
    plt.xlabel("lam")
    plt.ylabel("I_0")
    #point estimate:
    point_estimate = y_rescale( model3.forward(single_x)) 
    point_estimate = point_estimate.detach().cpu().numpy().flatten()
    plt.scatter(point_estimate[0],point_estimate[2],label="Point Estimate",color="black")
    single_y = single_y.detach().cpu().numpy()
    plt.scatter(single_y[0,0],single_y[0,2],label="True lam, I_0",color="red")
    plt.legend()
    plt.show()

In [ ]:
# heatmap for pair mu, I_0:
n = 12
X,Y = gen_xy_torch(n)
X = X.reshape((n,-1))
Ys = y_scaling(Y)
gaussians = torch.randn((n,3),device=device)
gen_new_ys = model_rnvp.reverse(gaussians,X)

for i in range(12):
    n_samples = 10000
    single_y = Y[i].reshape(1,-1)
    # gaussians = torch.randn((100,3),device=device)
    single_x = X[i].reshape(1,-1)
    print(single_y.shape,single_x.shape)
    samples = model_rnvp.sample(siVisualize the dierence between noise-free outcomes X and noisy outcomes ˜X to nd good ranges
for the new parameters and extend the prior psim(Y ) accordinglyngle_x,n_samples)
    samples = y_rescale(samples).detach().cpu().numpy()
    #print(samples.shape)
    
    plt.hist2d(samples[:,1],samples[:,2],bins=40)
    plt.xlabel("mu")
    plt.ylabel("I_0")
    #point estimate:
    point_estimate = y_rescale( model3.forward(single_x)) 
    point_estimate = point_estimate.detach().cpu().numpy().flatten()
    plt.scatter(point_estimate[1],point_estimate[2],label="Point Estimate",color="black")
    single_y = single_y.detach().cpu().numpy()
    plt.scatter(single_y[0,1],single_y[0,2],label="True mu, I_0",color="red")
    plt.legend()
    plt.show()

Comment: In the heatmaps we can see the influence of the noise, because the heatmaps are more often less sharp than the version without noise.

We now do the ex.4 from sheet 4a again but with the noisy simulation.

In [ ]:
def simulate_sir_simple(lam, mu, I_0, T: int = 100):
    dtype = np.float32

    lam = np.array(lam, dtype=dtype)
    mu  = np.array(mu,  dtype=dtype)

    batch_size = 1 if lam.shape == () else lam.shape[0]

    I = np.array(I_0, dtype=dtype).reshape((batch_size,))
    S = np.array([900] * batch_size, dtype=dtype)
    R = np.array([10]  * batch_size, dtype=dtype)

    N = S + I + R

    observations = np.zeros((batch_size, T, 2), dtype=dtype)
    amounts      = np.zeros((batch_size, T, 3), dtype=dtype)

    for i in range(T):
        delta_S = -lam * S * I / N
        delta_I =  lam * S * I / N - mu * I
        delta_R =  mu * I

        S += delta_S
        I += delta_I
        R += delta_R

        observations[:, i, :] = np.array([delta_S, delta_R]).T
        amounts[:, i, :]      = np.array((S, I, R)).T

    return observations, amounts


def gen_xy_noiseless(batch_size, T: int = 100):
    lam = np.random.uniform(0.01, 1.0, size=batch_size).astype(np.float32)
    mu  = np.random.uniform(0.01, 1.0, size=batch_size).astype(np.float32)
    I0  = np.random.randint(1, 50,  size=batch_size).astype(np.float32)

    observations, _ = simulate_sir_simple(
        lam=lam,
        mu=mu,
        I_0=I0,
        T=T
    )

    X = observations.reshape(batch_size, -1)           # (B, 2T)
    Y = np.stack([lam, mu, I0], axis=1)               # (B, 3)

    return X, Y

def y_scaling(Y):
    Y = Y.copy()
    Y[:, 0] = (Y[:, 0] - 0.01) / (1.0 - 0.01)  # lam in [0.01,1]
    Y[:, 1] = (Y[:, 1] - 0.01) / (1.0 - 0.01)  # mu  in [0.01,1]
    Y[:, 2] = Y[:, 2] / 50.0                   # I0 in [0,50]
    return torch.tensor(Y, dtype=torch.float32, device=device)

def y_unscale(Y_scaled):
    Y = Y_scaled.clone()
    Y[:, 0] = Y[:, 0] * (1.0 - 0.01) + 0.01
    Y[:, 1] = Y[:, 1] * (1.0 - 0.01) + 0.01
    Y[:, 2] = Y[:, 2] * 50.0
    return Y

def mmd_loss(x, y, bandwidths=None):
    batch_size = x.size(0)
    xx = torch.mm(x, x.t())
    yy = torch.mm(y, y.t())
    xy = torch.mm(x, y.t())

    rx = xx.diag().unsqueeze(0).expand_as(xx)
    ry = yy.diag().unsqueeze(0).expand_as(yy)

    K = 0
    L = 0
    P = 0
    if bandwidths is None:
        bandwidths = [0.4, 0.8, 1.6]

    for sigma in bandwidths:
        s = 1.0 / sigma**2
        K += 1.0 / (1.0 + s * (rx.t() + rx - 2.0 * xx))
        L += 1.0 / (1.0 + s * (ry.t() + ry - 2.0 * yy))
        P += 1.0 / (1.0 + s * (rx.t() + ry - 2.0 * xy))

        beta = 1.0 / (batch_size * (batch_size - 1) * len(bandwidths))
        gamma = 2.0 / (batch_size**2 * len(bandwidths))

    return beta * (torch.sum(K) + torch.sum(L)) - gamma * torch.sum(P)


def calculate_nllMMD_loss(model, z, hx, lambda_mmd=1.0):
    log_det = 0.0
    for block in model.realNVP.blocks:
        log_det += block.block_det

    nll = 0.5 * (z ** 2).sum(dim=1) - log_det
    nll = nll.mean()

    prior = torch.randn_like(hx)
    mmd = mmd_loss(hx, prior)

    return nll + lambda_mmd * mmd, nll, mmd


def train_inn_cond_mmd(
    model,
    train_set_fn,
    optim,
    epochs,
    batch_size=128,
    lambda_mmd=1.0
):
    model.train()
    scaler = GradScaler("cuda", enabled=(device.type == "cuda"))

    history = {
        "total": [],
        "nll": [],
        "mmd": []
    }

    for epoch in range(epochs):
        X, Y = train_set_fn(batch_size)

        X = torch.tensor(
            X.reshape(batch_size, -1),
            device=device,
            dtype=torch.float32
        )
        Y = y_scaling(Y)

        optim.zero_grad()

        # with torch.amp.autocast(
        #     "cuda",
        #     enabled=(device.type == "cuda")
        # ):
        hx = model.summary(X)
        z  = model.realNVP(Y, hx)

        loss, nll, mmd = calculate_nllMMD_loss(
            model,
            z,
            hx,
            lambda_mmd=lambda_mmd
        )

        #scaler.scale(loss).backward()
        #scaler.step(optim)
        #scaler.update()
        loss.backward()
        optim.step()

        history["total"].append(loss.item())
        history["nll"].append(nll.item())
        history["mmd"].append(mmd.item())

        print(
            f"Epoch {epoch+1:03d} | "
            f"Total {loss.item():.4f} | "
            f"NLL {nll.item():.4f} | "
            f"MMD {mmd.item():.4f}",
            end="\r"
        )

    print()
    return history


In [ ]:
model_mmd = RealNVPSummary().to(device)
optim_mmd = torch.optim.Adam(model_mmd.parameters(), lr=1e-3)
total_hist = {i:[] for i in ["total", "nll", "mmd"]}
history = train_inn_cond_mmd(
    model_mmd,
    gen_xy_noiseless,
    optim_mmd,
    epochs=500,
    batch_size=500,
    lambda_mmd=10.0
)
total_hist["mmd"].extend(history["mmd"])
total_hist["nll"].extend(history["nll"])
total_hist["total"].extend(history["total"])
plt.plot(total_hist["total"],label="total")
plt.plot(total_hist["nll"],label="nll")
plt.plot(total_hist["mmd"],label="mmd")

plt.legend()
plt.title("Total loss")
plt.yscale("log")
plt.show()

In [ ]:
for param_group in optim.param_groups:
    param_group["lr"] *= 0.1
history = train_inn_cond_mmd(
    model_mmd,
    gen_xy_noiseless,
    optim_mmd,
    epochs=500,
    batch_size=500,
    lambda_mmd=100.0
)
total_hist["mmd"].extend(history["mmd"])
total_hist["nll"].extend(history["nll"])
total_hist["total"].extend(history["total"])
plt.plot(total_hist["total"],label="total")
plt.plot(total_hist["nll"],label="nll")
plt.plot(total_hist["mmd"],label="mmd")

plt.title("Total loss")
plt.show()

In [ ]:
# chi2 on noisy
from scipy.stats import chi2

count = 0
n = 1000
for i in range(n):
    prior = sample_from_prior(1)
    lam = prior[0, 0]
    mu = prior[0, 1]
    I0 = prior[0, 2]
    L = prior[0, 3]
    rho = prior[0, 4]
    sigma_2 = prior[0, 5]

    noisy_obs = simulate_sir_noisy(
        lam=lam, mu=mu, I_0=I0, L=L, rho=rho, sigma_2=sigma_2, T=T, return_amounts=False
    )

    X = noisy_obs.reshape(1, -1)
    X_torch = torch.tensor(X, dtype=torch.float32, device=device)
    hx = model_mmd.summary(X_torch).detach().cpu().numpy()
    s2 = np.sum(hx**2)
    df = hx.size
    p_value = chi2.sf(s2, df=df)  # chi - square test
    p_const = 0.01
    if p_value <= p_const:
        count += 1
# count of p values lower than 0.001
print(
    f"Out of {n} simulations, the p value for the chi^2 test was lower than {p_const} in {count} simulations."
)

In [ ]:
# chi2 on unnoisy
from scipy.stats import chi2

count = 0
for i in range(n):
    prior = sample_from_prior(1)
    lam = prior[0, 0]
    mu = prior[0, 1]
    I0 = prior[0, 2]
    L = prior[0, 3]
    rho = prior[0, 4]
    sigma_2 = prior[0, 5]

    noisy_obs, _ = simulate_sir_simple(lam=lam, mu=mu, I_0=I0, T=T)

    X = noisy_obs.reshape(1, -1)
    X_torch = torch.tensor(X, dtype=torch.float32, device=device)
    hx = model_mmd.summary(X_torch).detach().cpu().numpy()
    s2 = np.sum(hx**2)
    df = hx.size
    p_value = chi2.sf(s2, df=df)  # chi - square test
    if p_value <= p_const:
        count += 1
# count of p values lower than 0.001
print(
    f"Out of {n} simulations, the p value for the chi^2 test was lower than {p_const} in {count} simulations."
)

In [ ]:
# ks on noisy
from scipy . stats import kstest 
count = 0
for i in range(n):
    prior = sample_from_prior(100)
    lam     = prior[:, 0]
    mu      = prior[:, 1]
    I0      = prior[:, 2]
    L       = prior[:, 3]
    rho     = prior[:, 4]
    sigma_2 = prior[:, 5]

    noisy_obs = simulate_sir_noisy(
                lam=lam, mu=mu, I_0=I0, L=L, rho=rho, sigma_2=sigma_2, T=T,return_amounts=False
            )

    X = noisy_obs.reshape(100, -1)
    X_torch = torch.tensor(X, dtype=torch.float32, device=device)
    hX = model_mmd.summary(X_torch).detach().cpu().numpy()
    S = hX # compute features for entire test set X ...
    S2 = np.sum ( S **2 , axis =1) # and their squared norm
    df = S.shape [1] # c al cu la te degrees of freedom
    stat , p_value = kstest( S2 , chi2 ( df ) . cdf ) # Kolmogorov - Smirnov test
    if p_value <= p_const:
        count += 1
print(f"Out of {n} simulations, the p value for the kolmogorov test was lower than {p_const} in {count} simulations.")

In [ ]:
#ks with no noise
count = 0
for i in range(n):
    prior = sample_from_prior(100)
    lam     = prior[:, 0]
    mu      = prior[:, 1]
    I0      = prior[:, 2]
    L       = prior[:, 3]
    rho     = prior[:, 4]
    sigma_2 = prior[:, 5]

    noisy_obs,_ = simulate_sir_simple(
                lam=lam, mu=mu, I_0=I0, T=T
            )

    X = noisy_obs.reshape(100, -1)
    X_torch = torch.tensor(X, dtype=torch.float32, device=device)
    hX = model_mmd.summary(X_torch).detach().cpu().numpy()
    S = hX # compute features for entire test set X ...
    S2 = np.sum ( S **2 , axis =1) # and their squared norm
    df = S.shape [1] # c al cu la te degrees of freedom
    stat , p_value = kstest( S2 , chi2 ( df ) . cdf ) # Kolmogorov - Smirnov test
    if p_value <= p_const:
        count += 1
print(f"Out of {n} simulations, the p value for the kolmogorov test was lower than {p_const} in {count} simulations.")

Comment: Both results cleary show, that the summary network trained on noiseless data does not map observations from the noisy simulation to a standard normal, which was what we expected.